# IS 362 - Project 2
## Cleaning and Analyzing Untidy Datasets

For this project, I selected three fictional datasets
from our Week 5 discussion board.

1. NBA Player Statistics - Elian Martinez
2. Public Library Usage - Joseph Damiano
3. Book Sales - My own discussion post

The goal of this project is to clean and reorganize
each dataset using pandas and then perform an analysis
to better understand the information.

In [1]:
import os

os.chdir("/Users/alyssaellis/Desktop/IS362_Project2")

print(os.listdir())

['library_usage.csv', 'book_sales.csv', 'nba_stats.csv', 'IS362_Project2.ipynb']


In [2]:
import pandas as pd

nba = pd.read_csv("nba_stats.csv")

nba

,Player,Team,Game Stats,Shooting
0,Marcus Reed,New York,"24 PTS, 7 REB, 5 AST",9/17 FG
1,Jordan Ellis,Miami,"18 PTS, 11 REB, 3 AST",7/14 FG
2,Devin Carter,Boston,"29 PTS, 4 REB, 8 AST",11/20 FG
3,Andre Lewis,Chicago,"21 PTS, 6 REB, 7 AST",8/16 FG


In [3]:
# Separate the game statistics into individual columns
nba[["Points", "Rebounds", "Assists"]] = (
    nba["Game Stats"]
    .str.extract(r"(\d+) PTS, (\d+) REB, (\d+) AST")
    .astype(int)
)

# Separate field goals made and attempted
nba[["FG Made", "FG Attempted"]] = (
    nba["Shooting"]
    .str.extract(r"(\d+)/(\d+) FG")
    .astype(int)
)

# Calculate field goal percentage
nba["FG Percentage"] = (
    nba["FG Made"] / nba["FG Attempted"] * 100
).round(1)

# Display the cleaned dataset
nba_clean = nba[["Player", "Team", "Points",
                 "Rebounds", "Assists", "FG Made",
                 "FG Attempted", "FG Percentage"]]

nba_clean

,Player,Team,Points,Rebounds,Assists,FG Made,FG Attempted,FG Percentage
0,Marcus Reed,New York,24,7,5,9,17,52.9
1,Jordan Ellis,Miami,18,11,3,7,14,50.0
2,Devin Carter,Boston,29,4,8,11,20,55.0
3,Andre Lewis,Chicago,21,6,7,8,16,50.0


### Cleaning the NBA Dataset

The original dataset had multiple statistics combined into single columns, which made it harder to analyze. I used pandas to separate the points, rebounds, and assists into their own columns. I also separated the shooting statistics into field goals made and attempted.

After cleaning the data, I calculated each player's field goal percentage. This made it easier to compare the players and look at their individual performances.

In [4]:
# Compare players by points scored
points_ranked = nba_clean.sort_values(
    by="Points", ascending=False
)

print("Players Ranked by Points:")
display(points_ranked[["Player", "Points"]])

# Calculate average statistics
print("Average Player Statistics:")
display(
    nba_clean[["Points", "Rebounds", "Assists",
               "FG Percentage"]].mean().round(2)
)

# Find the player with the highest shooting percentage
best_shooter = nba_clean.loc[
    nba_clean["FG Percentage"].idxmax()
]

print("Highest Field Goal Percentage:")
print(best_shooter["Player"], "-", best_shooter["FG Percentage"], "%")

Players Ranked by Points:


,Player,Points
2,Devin Carter,29
0,Marcus Reed,24
3,Andre Lewis,21
1,Jordan Ellis,18


Average Player Statistics:


Points           23.00
Rebounds          7.00
Assists           5.75
FG Percentage    51.98
dtype: float64

Highest Field Goal Percentage:
Devin Carter - 55.0 %


### NBA Analysis and Conclusions

After cleaning the dataset, I compared the players based on points, rebounds, assists, and shooting percentage.

Devin Carter scored the most points with 29 and also had the highest field goal percentage at 55%. Jordan Ellis had the most rebounds with 11, while Devin Carter led in assists with 8.

The average points scored across the four players was 23. Overall, cleaning the dataset made it easier to compare individual player statistics and identify which players performed best in different categories.

One limitation is that the original dataset only includes one set of statistics per player, so I couldn't compare performance between different seasons.

## Dataset 2: Public Library Usage

**Original Dataset by Joseph Damiano**

This fictional dataset shows monthly library usage across three branches: Downtown, Northside, and West End. It includes the number of books borrowed and computer sessions for four different age groups.

The original dataset is untidy because the age groups are stored as separate columns instead of being organized into one age group column. I will use pandas to reorganize the data and compare how different age groups use library services.

In [5]:
# Import the library dataset
library = pd.read_csv("library_usage.csv")

# Display the original data
library

,Branch,Service Type,Kids (0-12),Teens (13-18),Adults (19-64),Seniors (65+)
0,Downtown,Books Borrowed,520,180,940,610
1,Downtown,Computer Sessions,110,320,580,210
2,Northside,Books Borrowed,340,115,620,450
3,Northside,Computer Sessions,85,210,390,140
4,West End,Books Borrowed,210,90,430,380
5,West End,Computer Sessions,45,160,290,95


In [6]:
# Convert the wide dataset into a tidy format
library_clean = pd.melt(
    library,
    id_vars=["Branch", "Service Type"],
    var_name="Age Group",
    value_name="Usage Count"
)

# Display the cleaned dataset
library_clean

,Branch,Service Type,Age Group,Usage Count
0,Downtown,Books Borrowed,Kids (0-12),520
1,Downtown,Computer Sessions,Kids (0-12),110
2,Northside,Books Borrowed,Kids (0-12),340
3,Northside,Computer Sessions,Kids (0-12),85
4,West End,Books Borrowed,Kids (0-12),210
5,West End,Computer Sessions,Kids (0-12),45
6,Downtown,Books Borrowed,Teens (13-18),180
7,Downtown,Computer Sessions,Teens (13-18),320
8,Northside,Books Borrowed,Teens (13-18),115
9,Northside,Computer Sessions,Teens (13-18),210


### Cleaning the Library Dataset

The original library dataset had separate columns for kids, teens, adults, and seniors. This made it harder to compare the age groups and analyze their usage.

I used the pandas melt() function to change the dataset from wide format to long format. Instead of having a separate column for each age group, I created an Age Group column and a Usage Count column.

The cleaned dataset now has 24 rows, with each row representing a specific library branch, service type, and age group. This makes it easier to group and compare the data.

In [7]:
# Compare total usage by age group and service type
usage_by_age = library_clean.pivot_table(
    index="Age Group",
    columns="Service Type",
    values="Usage Count",
    aggfunc="sum"
)

print("Library Usage by Age Group:")
display(usage_by_age)

# Compare total usage across library branches
usage_by_branch = library_clean.groupby(
    "Branch"
)["Usage Count"].sum().sort_values(ascending=False)

print("Total Usage by Branch:")
display(usage_by_branch)

# Find total computer sessions by age group
computer_usage = library_clean[
    library_clean["Service Type"] == "Computer Sessions"
].groupby("Age Group")["Usage Count"].sum().sort_values(ascending=False)

print("Computer Sessions by Age Group:")
display(computer_usage)

Library Usage by Age Group:


Service Type,Books Borrowed,Computer Sessions
Age Group,,
Adults (19-64),1990,1260
Kids (0-12),1070,240
Seniors (65+),1440,445
Teens (13-18),385,690


Total Usage by Branch:


Branch
Downtown     3470
Northside    2350
West End     1700
Name: Usage Count, dtype: int64

Computer Sessions by Age Group:


Age Group
Adults (19-64)    1260
Teens (13-18)      690
Seniors (65+)      445
Kids (0-12)        240
Name: Usage Count, dtype: int64

### Library Analysis and Conclusions

After cleaning the dataset, I compared library usage across different age groups and branches.

Adults had the highest overall usage, with 1,990 books borrowed and 1,260 computer sessions. Teens used computers more often than they borrowed books, with 690 computer sessions compared to 385 books borrowed.

Downtown had the highest total usage out of the three branches, followed by Northside and West End.

Based on these results, the library could consider focusing computer resources on teens and adults while continuing to provide book collections for all age groups. This analysis shows how reorganizing the dataset makes it easier to compare library services and identify usage patterns.

## Dataset 3: Book Sales

**Original Dataset: My Week 5 Discussion Post**

For my dataset, I created fictional sales data for five books at a small bookstore over three months: January, February, and March.

The original dataset is untidy because each month has its own column instead of having one column for the month and another for sales. I will use pandas to reorganize the data and analyze which books and genres sold the most, along with how sales changed over time.

In [8]:
# Import the bookstore sales dataset
books = pd.read_csv("book_sales.csv")

# Display the original dataset
books

,Book Title,Genre,January Sales,February Sales,March Sales
0,The Last Summer,Romance,45,52,61
1,Midnight House,Horror,32,41,38
2,Lost in the Stars,Science Fiction,27,35,46
3,The Silent Case,Mystery,51,48,57
4,Second Chances,Romance,39,44,50


In [9]:
# Convert monthly sales columns into rows
books_clean = pd.melt(
    books,
    id_vars=["Book Title", "Genre"],
    var_name="Month",
    value_name="Sales"
)

# Remove " Sales" from the month names
books_clean["Month"] = books_clean["Month"].str.replace(
    " Sales", "", regex=False
)

# Organize the months in chronological order
books_clean["Month"] = pd.Categorical(
    books_clean["Month"],
    categories=["January", "February", "March"],
    ordered=True
)

# Sort by book title and month
books_clean = books_clean.sort_values(
    by=["Book Title", "Month"]
).reset_index(drop=True)

books_clean

,Book Title,Genre,Month,Sales
0,Lost in the Stars,Science Fiction,January,27
1,Lost in the Stars,Science Fiction,February,35
2,Lost in the Stars,Science Fiction,March,46
3,Midnight House,Horror,January,32
4,Midnight House,Horror,February,41
5,Midnight House,Horror,March,38
6,Second Chances,Romance,January,39
7,Second Chances,Romance,February,44
8,Second Chances,Romance,March,50
9,The Last Summer,Romance,January,45


### Cleaning the Book Sales Dataset

The original dataset had separate columns for January, February, and March sales. This made it more difficult to compare sales over time.

I used the pandas melt() function to transform the data from wide format into long format. I combined the monthly sales columns into a Month column and a Sales column.

I also removed the word "Sales" from the month names and organized the months in chronological order. The cleaned dataset now contains 15 rows, making it easier to compare books, genres, and monthly sales.

In [10]:
# Find total sales for each book
sales_by_book = books_clean.groupby(
    "Book Title"
)["Sales"].sum().sort_values(ascending=False)

print("Total Sales by Book:")
display(sales_by_book)

# Find total sales by genre
sales_by_genre = books_clean.groupby(
    "Genre"
)["Sales"].sum().sort_values(ascending=False)

print("Total Sales by Genre:")
display(sales_by_genre)

# Compare sales across the three months
monthly_sales = books_clean.groupby(
    "Month", observed=True
)["Sales"].sum()

print("Total Sales by Month:")
display(monthly_sales)

# Compare the two romance books
romance_sales = books_clean[
    books_clean["Genre"] == "Romance"
].pivot_table(
    index="Month",
    columns="Book Title",
    values="Sales",
    aggfunc="sum",
    observed=True
)

print("Romance Book Comparison:")
display(romance_sales)

Total Sales by Book:


Book Title
The Last Summer      158
The Silent Case      156
Second Chances       133
Midnight House       111
Lost in the Stars    108
Name: Sales, dtype: int64

Total Sales by Genre:


Genre
Romance            291
Mystery            156
Horror             111
Science Fiction    108
Name: Sales, dtype: int64

Total Sales by Month:


Month
January     194
February    220
March       252
Name: Sales, dtype: int64

Romance Book Comparison:


Book Title,Second Chances,The Last Summer
Month,,
January,39,45
February,44,52
March,50,61


### Book Sales Analysis and Conclusions

After cleaning the dataset, I compared the total sales for each book, genre, and month.

The Last Summer had the highest total sales with 158 books sold, followed closely by The Silent Case with 156. Romance was the most popular genre overall, with 291 books sold across the two romance titles.

I also found that total bookstore sales increased each month, going from 194 in January to 220 in February and 242 in March.

When comparing the two romance books, The Last Summer sold more copies than Second Chances in all three months.

Overall, reorganizing the dataset made it easier to see sales trends and compare different books. Based on these results, the bookstore might consider stocking more romance books, since they had the highest combined sales in this sample.